# **Text Representation Techniques for RAG Systems**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Introduction to Text Representation: Bag-of-Words Model](#Introduction-to-Text-Representation:-Bag-of-Words-Model)
  - [Introduction](#Introduction)
  - [Why Text Representation is Essential](#Why-Text-Representation-is-Essential)
  - [Understanding BOW Model](#Understanding-BOW-Model)
  - [Building a Basic Vocabulary](#Building-a-Basic-Vocabulary)
  - [Converting Text to Vectors](#Converting-Text-to-Vectors)
  - [Conclusion and Next Steps](#Conclusion-and-Next-Steps)
  - [Text Cleaning with Python](#Text-Cleaning-with-Python)
  - [Building a Vocabulary Dictionary](#Building-a-Vocabulary-Dictionary)
  - [Transform Text into Numeric Vectors](#Transform-Text-into-Numeric-Vectors)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Introduction to Text Representation: Bag-of-Words Model

## Introduction

Welcome to the very first lesson of our course "**Text Representation Techniques for RAG Systems**", part of our "**Foundations of RAG Systems**" course path! In the first course of this learning path, you learned the fundamentals of **RAG**, how to structure a simple **RAG workflow**, and why combining retrieval with generation is so powerful. Now, we'll shift our focus to how we can turn raw text into numerical data — a crucial step if we want our RAG systems to retrieve information accurately and feed it into downstream pipelines; in other words, we'll focus on the **indexing** component of our RAG pipeline.

In this lesson, our main objectives are:
1. Understand why we must transform text into a structured format for RAG workflows.
2. Explore the **Bag-of-Words (BOW)** method, a simple yet classic text representation technique.

By the end, you'll know how words get mapped into vectors and why these representations matter when building robust retrieval systems.

## Why Text Representation is Essential

RAG systems revolve around retrieving relevant documents based on a user’s query, then generating a final answer. However, computers don’t process language the way humans do; they require structured or numerical forms of text to effectively compare one document with another. Without a proper representation of text, two main issues arise:

* We *can’t reliably measure how similar* one piece of text is to another.
* It becomes far more difficult to retrieve accurate, *contextually relevant* information.

A straightforward solution to this challenge is the Bag-of-Words method. It works by counting how often each word appears, providing a simple numerical snapshot of a document. While this approach ignores the order of words and misses linguistic nuances, it’s an excellent entry point for understanding how to convert messy human language into machine-friendly formats that form the core of RAG systems.

## Understanding BOW Model

Let’s explore how Bag-of-Words vectors capture word frequency without considering word order. Consider these three sentences:

1. “I love machine learning”
2. “Machine learning is fun”
3. “I love coding”

To construct our BOW representation, we first gather all unique words to form our vocabulary: `{I, love, machine, learning, is, fun, coding}`. Each word in the vocabulary maps to an index:

| Word  | I   | love | machine | learning | is  | fun | coding |
| :---- | :-: | :--: | :-----: | :------: | :-: | :-: | :----: |
| Index | 0   | 1    | 2       | 3        | 4   | 5   | 6      |

With this vocabulary, we can transform each sentence into a numeric vector by counting the occurrences of each word. For example:


| Sentence | I   | love | machine | learning | is  | fun | coding |
| :------- | :-: | :--: | :-----: | :------: | :-: | :-: | :----: |
| I love machine learning | 1 | 1 | 1 | 1 | 0 | 0 | 0 |
| Machine learning is fun | 0 | 0 | 1 | 1 | 1 | 1 | 0 |
| I love coding | 1 | 1 | 0 | 0 | 0 | 0 | 1 |

In these vectors, each column corresponds to a word in the vocabulary, and the numbers indicate how often each word appears in the sentence. This frequency-based representation is a straightforward way to convert text into numerical form for retrieval tasks, providing a foundational understanding of text representation.

## Building a Basic Vocabulary

In a BOW approach, the first step is constructing a dictionary (often called a “vocabulary”) that assigns each distinct word an integer index. Here's how you might do it using a function that takes as input a collection of documents:

In [ ]:
def build_vocab(docs):
    unique_words = set()
    for doc in docs:
        for word in doc.lower().split():
            # Strip punctuation around words (simple approach)
            clean_word = word.strip(".,?!")
            if clean_word:
                unique_words.add(clean_word)
    # Sort the unique_words to have a consistent, deterministic order
    return {word: idx for idx, word in enumerate(sorted(unique_words))}

Let's see what's happening:

1. We iterate through each text in `docs`.
2. We convert the text to lowercase and split it into tokens (words).
3. We remove punctuation at the start and end of these tokens.
4. Each cleaned token is added to a `unique_words` set so we only keep distinct words.
5. Finally, we sort the collection of `unique words` to ensure a consistent order and assign each word an index stored in a dictionary.

This vocabulary allows us to look up a word and pinpoint exactly where it should appear in any BOW vector.

## Converting Text to Vectors

Once you have a vocabulary, you can create a numeric BOW vector for each text. Every position in the vector corresponds to one word in the vocabulary, and the vector entries indicate how many times each vocabulary word appears in the text. Let's illustrate with code:

In [ ]:
import numpy as np

def bow_vectorize(text, vocab):
    vector = np.zeros(len(vocab), dtype=int)
    for word in text.lower().split():
        clean_word = word.strip(".,?!")
        if clean_word in vocab:
            # Increment the vector slot corresponding to this word
            vector[vocab[clean_word]] += 1
    return vector

1. We create an `np` array of zeros, with the length equal to the number of unique words in our vocabulary.
2. For each cleaned token in our text, we look up its vocabulary index.
3. We increment the vector at that index, effectively counting the occurrences of each vocabulary word.

Even though BOW ignores word order and deeper linguistic information, it remains a valuable technique for smaller-scale tasks or as a baseline representation. It demonstrates the mechanics of transforming words into arrays of numerical counts, which is essential for retrieval tasks.

## Conclusion and Next Steps

In this lesson, you learned why text must be represented numerically for RAG systems and walked through a foundational technique — **Bag-of-Words** — to convert words into count-based vectors. While BOW has limitations in capturing context, it's an excellent first step in any NLP workflow. Practice building vocabularies and generating BOW vectors with your own text to get comfortable with these concepts.

Up next, we'll explore more advanced methods that preserve the sense and context of your text, such as embeddings derived from language models. In the practice exercises that follow, you'll get hands-on experience coding your own BOW pipeline.

This foundation will set the stage for more powerful semantic retrieval techniques and deeper integration with RAG pipelines later in the course. Good luck, and have fun experimenting!

## Text Cleaning with Python

You've just learned how to build a vocabulary and convert text into Bag-of-Words vectors. Now, let's put that knowledge into practice with a simple task.

Your objective is to create a function called `preprocess_string` that processes a single string. Here's what you need to do:

* Split the string into words.
* Convert each word to lowercase.
* Remove punctuation from the start and end of each word (bonus points for doing it with a list comprehension!).

This exercise will help you solidify your understanding of text preprocessing. Dive in and see how well you can clean up a string!

In [ ]:
def preprocess_string(text):
    # TODO: Split the text into words
    # TODO: Convert words to lowercase and remove punctuation
    # TODO: Return the cleaned tokens
    words = text.split()
    cleaned_words = [word.lower().strip(".,?!") for word in words]
    return cleaned_words

In [ ]:
print(preprocess_string("Hello, World! We are preprocessing strings today."))

['hello', 'world', 'we', 'are', 'preprocessing', 'strings', 'today']


## Building a Vocabulary Dictionary

Nice job on processing unique words and their counts! Now, let's take it a step further by creating a *vocabulary dictionary*.

Your task is to build a function that:

* Takes a list of tokens.
* Sorts these words.
* Assigns each word a numeric index to form a vocabulary dictionary.

This exercise will help you understand how to map words to indices, a key step in text representation. Dive in and see how well you can create a structured vocabulary!

In [ ]:
def preprocess_string(text):
    words = text.split()
    cleaned_tokens = [word.lower().strip(".,!?") for word in words]
    return cleaned_tokens


def build_vocab(tokens):
    # TODO: Sort the unique tokens and assign each a numeric index
    unique_tokens = set(tokens)
    return {token: idx for idx, token in enumerate(sorted(unique_tokens))}

In [ ]:
sentence = "Hello, World! We are preprocessing strings today."
tokens = preprocess_string(sentence)
vocab = build_vocab(tokens)
print(f'Vocabulary:\n{vocab}')

Vocabulary:
{'are': 0, 'hello': 1, 'preprocessing': 2, 'strings': 3, 'today': 4, 'we': 5, 'world': 6}


## Transform Text into Numeric Vectors

You've just explored how to build a vocabulary and convert text into *Bag-of-Words* vectors. Now, let's apply that knowledge in a practical task.

Your objective is to complete the `bow_vectorize` function. Here's what you need to do:

* Create a zero vector with the same length as the vocabulary.
* For each word in the text, clean it and check if it's in the vocabulary.
* Increment the vector slot corresponding to each vocabulary word found.

This exercise will help you solidify your understanding of transforming text into numeric vectors. Dive in and see how well you can implement this transformation!

In [ ]:
import numpy as np

def build_vocab(docs):
    unique_words = set()
    for doc in docs:
        for word in doc.lower().split():
            clean_word = word.strip(".,?!")
            if clean_word:
                unique_words.add(clean_word)
    return {word: idx for idx, word in enumerate(sorted(unique_words))}

def bow_vectorize(text, vocab):
    # TODO: Create a zero vector with the same length as the vocabulary
    # TODO: Convert the text to lowercase and split it into words
    # TODO: For each word, clean it by stripping punctuation
    # TODO: Check if the cleaned word is in the vocabulary
    # TODO: If the word is in the vocabulary, increment the corresponding index in the vector
    vector = np.zeros(len(vocab), dtype=int)
    for word in text.lower().split():
        clean_word = word.strip(".,?!")
        if clean_word in vocab:
            vector[vocab[clean_word]] += 1
    return vector

In [ ]:
example_texts = [
    "RAG stands for retrieval augmented generation, and retrieval is a key component of RAG.",
    "Data is crucial for retrieval processes, and without data, retrieval systems cannot function effectively."
]

vocab = build_vocab(example_texts)
print(f'Vocabulary:\n{vocab.items()}')

for text in example_texts:
    vector = bow_vectorize(text, vocab)
    print(f'Text: {text}\nBOW Vector: {vector}\n')

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[CodeSignal: Text Representation Techniques for RAG Systems Course](https://codesignal.com/learn/courses/text-representation-techniques-for-rag-systems)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Figure 1: Word Embeddings</em>
    </td>
  </tr>
</table>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)